## Riadattamento Deep Models

### SETUP

In [ ]:
from data_pipeline import *

#split and clean
df = load_data('data/data-ready-def.xlsx')
train_df, valid_df, test_df = split_data(df)  #i tre set splittati

train_df_clean = clean_data(train_df)
valid_df_clean = clean_data(valid_df)
test_df_clean = clean_data(test_df)

#normalize
train_df_norm, valid_df_norm, test_df_norm, train_df_min, train_df_max = normalize(train_df_clean, valid_df_clean, test_df_clean)

#create_sequence con H = 1
train_X, train_y = create_sequences(train_df_norm, 20, 1)
print(train_X.shape)
print(train_y.shape)

valid_X, valid_y = create_sequences(valid_df_norm, 20, 1)
print(valid_X.shape)
print(valid_y.shape)

test_X, test_y = create_sequences(test_df_norm, 20, 1)
print(test_X.shape)
print(test_y.shape)

In [ ]:
#librerie modelli deep
import random
import tensorflow as tf
import keras
from keras.layers import Input, Dense, LSTM, Conv1D, Flatten, Bidirectional, BatchNormalization, Activation, add, Permute, Multiply
from keras.models import Model
from keras.callbacks import EarlyStopping

from baselines import evaluate

In [ ]:
#funzione di addestramento e valutazione modelli
def fit_and_evaluate_deep(model, train_X, train_y, valid_X, valid_y, test_X, test_y, target_min, target_max, epochs=100, batch_size=64, patience=15):
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')

  #verificato Overfitting, strategia per evitarlo
  #la patience misura il peso che il modello dà all'insieme di rumore, più è alto più il modello ha uno sguardo ampio (15 alto)
  early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

  #serve per i grafici
  history = model.fit(train_X, train_y, validation_data=(valid_X, valid_y), epochs=epochs, batch_size=batch_size, callbacks=[early_stop])

  #appiattisco e salvo le previsioni (3D -> 2D)
  valid_pred_norm = model.predict(valid_X).flatten()
  test_pred_norm = model.predict(test_X).flatten()

  #denormalizzo previsioni
  valid_pred_mm = valid_pred_norm * (target_max - target_min) + target_min
  test_pred_mm = test_pred_norm * (target_max - target_min) + target_min

  #clip: l'irrigazione non può essere negativa
  valid_pred_mm = np.clip(valid_pred_mm, a_min=0, a_max=None)
  test_pred_mm = np.clip(test_pred_mm, a_min=0, a_max=None)

  #denormalizzo target (già appiattiti)
  valid_true_mm = valid_y * (target_max - target_min) + target_min
  test_true_mm = test_y * (target_max - target_min) + target_min  

  #calcolo metriche, utilizzo la funzione evaluate()
  valid_mae, valid_rmse, valid_r2 = evaluate(valid_true_mm, valid_pred_mm)
  test_mae, test_rmse, test_r2 = evaluate(test_true_mm, test_pred_mm)

  return valid_mae, valid_rmse, valid_r2, test_mae, test_rmse, test_r2, history


### LSTM

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def build_lstm_model(input_dims, time_steps, lstm_units): #corrispettiva di Main2_listm.attention_model del paper
  
  inputs = Input(shape=(time_steps, input_dims))
  lstm_out = LSTM(lstm_units, activation='relu')(inputs)
  output = Dense(1)(lstm_out)
  model = Model(inputs=[inputs], outputs=output)
  
  return model

m = build_lstm_model(train_X.shape[2], train_X.shape[1], lstm_units=64)
#le 64 unità favoriscono una riduzione di "memorizzazione del rumore" da parte del modello, 'modelli piccoli nella proposta'

m.summary()

lstm_valid_mae, lstm_valid_rmse, lstm_valid_r2, lstm_test_mae, lstm_test_rmse, lstm_test_r2, history = fit_and_evaluate_deep(m, train_X, train_y, valid_X,
valid_y, test_X, test_y, train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation'])

print(lstm_valid_mae, lstm_valid_rmse, lstm_valid_r2)
print(lstm_test_mae, lstm_test_rmse, lstm_test_r2)


### CNN

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def build_cnn_model(input_dims, time_steps, filters):

  inputs = Input(shape=(time_steps, input_dims))
  x = Conv1D(filters=filters, kernel_size=1, activation='relu')(inputs)
  attention_mul = Flatten()(x)
  output = Dense(1)(attention_mul)
  model = Model(inputs=[inputs], outputs=output)
  
  return model

m = build_cnn_model(train_X.shape[2], train_X.shape[1], filters=64)

m.summary()

cnn_valid_mae, cnn_valid_rmse, cnn_valid_r2, cnn_test_mae, cnn_test_rmse, cnn_test_r2, history = fit_and_evaluate_deep(m, train_X, train_y, valid_X,
valid_y, test_X, test_y, train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation'])

print(cnn_valid_mae, cnn_valid_rmse, cnn_valid_r2)
print(cnn_test_mae, cnn_test_rmse, cnn_test_r2)

### BiLSTM-CNN-Attention

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def build_bilstm_block_test(input_dims, time_steps, lstm_units):
  
  inputs = Input(shape=(time_steps, input_dims))
  lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True))(inputs)
  model = Model(inputs=[inputs], outputs=lstm_out)
  
  return model

m = build_bilstm_block_test(train_X.shape[2], train_X.shape[1], lstm_units=64)

m.summary()
  

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def build_bilstm_block_test2(input_dims, time_steps, lstm_units):
  
  inputs = Input(shape=(time_steps, input_dims))
  lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True))(inputs)

  x2 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(lstm_out)

  x3 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x2)
  x3 = BatchNormalization()(x3)
  x3 = Conv1D(filters=64, kernel_size=3, padding='same')(x3)
  x3 = BatchNormalization()(x3)
  x3 = add([x2, x3])
  x3 = Activation('relu')(x3)

  model = Model(inputs=[inputs], outputs=x3)
  return model

m_test = build_bilstm_block_test2(train_X.shape[2], train_X.shape[1], lstm_units=64)
m_test.summary()
  

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def attention_3d_block(inputs):
    
    input_dim = int(inputs.shape[2])
    a = inputs
    a = Dense(input_dim, activation='softmax')(a)
    a_probs = Permute((1, 2))(a)
    output_attention_mul = Multiply()([inputs, a_probs])
    
    return output_attention_mul

def build_bilstm_block_test3(input_dims, time_steps, lstm_units):
  
  inputs = Input(shape=(time_steps, input_dims))
  lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True))(inputs)

  x2 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(lstm_out)

  x3 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x2)
  x3 = BatchNormalization()(x3)
  x3 = Conv1D(filters=64, kernel_size=3, padding='same')(x3)
  x3 = BatchNormalization()(x3)
  x3 = add([x2, x3])
  x3 = Activation('relu')(x3)

  attention_mul = attention_3d_block(x3)

  model = Model(inputs=[inputs], outputs=attention_mul)
  return model

m_test = build_bilstm_block_test3(train_X.shape[2], train_X.shape[1], lstm_units=64)
m_test.summary()

In [17]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

def build_bilstm_cnn_attention_model(input_dims, time_steps, lstm_units):
  
  inputs = Input(shape=(time_steps, input_dims))

  lstm_out = Bidirectional(LSTM(lstm_units, return_sequences=True))(inputs)

  x2 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(lstm_out)

  x3 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x2)
  x3 = BatchNormalization()(x3)
  x3 = Conv1D(filters=64, kernel_size=3, padding='same')(x3)
  x3 = BatchNormalization()(x3)
  x3 = add([x2, x3])
  x3 = Activation('relu')(x3)

  attention_mul = attention_3d_block(x3)
  attention_mul = Flatten()(attention_mul)
  output = Dense(1)(attention_mul)

  model = Model(inputs=[inputs], outputs=output)
  
  return model

m = build_bilstm_cnn_attention_model(train_X.shape[2], train_X.shape[1], lstm_units=64)

m.summary()

bilstm_cnn_attn_valid_mae, bilstm_cnn_attn_valid_rmse, bilstm_cnn_attn_valid_r2, bilstm_cnn_attn_test_mae, bilstm_cnn_attn_test_rmse, bilstm_cnn_attn_test_r2, history = fit_and_evaluate_deep(
    m, train_X, train_y, valid_X, valid_y, test_X, test_y,
    train_df_min['Amount of irrigation'], train_df_max['Amount of irrigation']
)

print(bilstm_cnn_attn_valid_mae, bilstm_cnn_attn_valid_rmse, bilstm_cnn_attn_valid_r2)
print(bilstm_cnn_attn_test_mae, bilstm_cnn_attn_test_rmse, bilstm_cnn_attn_test_r2)

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 20, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 20, 128)   │     37,376 │ input_layer_10[0… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 20, 64)    │     24,640 │ bidirectional_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 20, 64)    │     12,352 │ conv1d_14[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 64)    │        256 │ conv1d_15[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 20, 64)    │     12,352 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 64)    │        256 │ conv1d_16[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 20, 64)    │          0 │ conv1d_14[0][0],  │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 20, 64)    │          0 │ add_4[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 20, 64)    │      4,160 │ activation_4[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute_3 (Permute) │ (None, 20, 64)    │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 20, 64)    │          0 │ activation_4[0][… │
│ (Multiply)          │                   │            │ permute_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 1280)      │          0 │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │      1,281 │ flatten_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 92,673 (362.00 KB)

 Trainable params: 92,417 (361.00 KB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 60ms/step - loss: 0.0328 - val_loss: 0.0644
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0285 - val_loss: 0.0631
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0281 - val_loss: 0.0674
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0273 - val_loss: 0.0659
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0269 - val_loss: 0.0669
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0265 - val_loss: 0.0667
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0260 - val_loss: 0.0641
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0256 - val_loss: 0.0600
Epoch 9/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0252 - val_loss: 0.0582
Epoch 10/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0249 - val_loss: 0.0510
Epoch 11/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0246 - val_loss: 0.0498
Epoch 12/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/ste